In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.neighbors import LocalOutlierFactor
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
import os
import time

In [ ]:
# Constants
RANDOM_STATE = 1082
np.random.seed(RANDOM_STATE)
TRAIN_SIZE = 0.2
DATA_PATH = "../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv"
MODEL_NAME = "LocalOutlierFactor"
DATASET_NAME = "spoofing-merged-gps-only.csv"
RESULTS_CSV = "../../../UAVATTACK_Results.csv"

In [ ]:
# Load the dataset
try:
    df = pd.read_csv("../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv")
except FileNotFoundError:
    print("Error: The file '../../../Datasets/csv/UAVATTACK/spoofing-merged-gps-only.csv' was not found.")
    exit()

In [ ]:
# Preprocess the data

# Clean up column names
df.columns = df.columns.str.replace('[\\[\\]<>]', '_', regex=True)

# Split features and label
X = df.drop(columns=['timestamp', 'label'])
y = df['label']

# Encode labels
if y.dtype == object or y.dtype.name == 'category':
    le = LabelEncoder()
    y = le.fit_transform(y)

# Split benign and malicious
benign_mask = y == 0
X_benign = X[benign_mask]
y_benign = y[benign_mask]

# Training set: benign only
X_train_raw, X_benign_val_raw, y_train, y_benign_val = train_test_split(
    X_benign, y_benign, test_size=TRAIN_SIZE, random_state=RANDOM_STATE
)

# Scale features using the benign training split only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_benign_val = scaler.transform(X_benign_val_raw)

# Test set: all samples (benign and malicious)
X_test = scaler.transform(X)
y_test = y


In [ ]:
# Train Local Outlier Factor model
lof = LocalOutlierFactor(n_neighbors=60, algorithm='auto',novelty=True, contamination='auto')
start_time = time.time()
lof.fit(X_train)
train_time = time.time() - start_time

In [ ]:
# Predict on test set: -1 is outlier, 1 is inlier
y_pred_lof = lof.predict(X_test)
y_pred = (y_pred_lof == -1).astype(int)  # 1 for anomaly, 0 for benign

In [ ]:
# Results
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f"Model Accuracy on Test Set: {accuracy * 100:.2f}%")
print(f"F1 Score: {f1:.4f}")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

In [ ]:
import os as _os
import sys as _sys

_root = _os.path.abspath(_os.getcwd())
while _root != _os.path.dirname(_root) and not _os.path.isfile(_os.path.join(_root, "eval_metrics.py")):
    _root = _os.path.dirname(_root)
if _root not in _sys.path:
    _sys.path.insert(0, _root)
from eval_metrics import append_row, evaluate, summary, time_inference


_proba = None
_timing = None

row = evaluate(
    y_test,
    y_pred,
    model_name=MODEL_NAME,
    dataset_name=DATASET_NAME,
    family="OneClass",
    labels=[0, 1],
    class_names=["benign", "malicious"],
    y_proba=_proba,
    y_train=y_train,
    train_time_sec=train_time,
    num_epochs="N/A",
    parameters=lof.get_params(),
    seed=RANDOM_STATE,
    n_features=None,
    n_train=len(y_train),
    test_set="real_full_incl_train_benign",
    timing=_timing,
)
append_row(row, RESULTS_CSV)
print(summary(row))
print(f"Results appended to {RESULTS_CSV}")